Playoff cleaning

In [5]:
import re
import pandas as pd

# Read extracted playoff stats and regular-season salary data
playoff_stats = pd.read_csv('data/matched_playoff_stats_2025.csv', encoding='utf-8-sig')
regular_salary = pd.read_csv('data/regular_salary.csv', encoding='latin1')


def find_name_column(df, label):
    candidates = [
        'Player', 'PLAYER', 'player',
        'Name', 'NAME', 'name',
        'Player Name', 'player_name', 'PlayerName'
    ]
    for c in candidates:
        if c in df.columns:
            return c
    for c in df.columns:
        low = str(c).strip().lower()
        if 'player' in low or low == 'name':
            return c
    raise ValueError(f'Could not find player-name column in {label}. Columns: {list(df.columns)}')


def normalize_player_name(name):
    if pd.isna(name):
        return ''
    s = str(name).lower().strip()
    s = re.sub(r"\b(jr|sr|ii|iii|iv|v)\b\.?,?", '', s)
    s = re.sub(r"[^a-z\s]", '', s)
    s = re.sub(r"\s+", ' ', s).strip()
    return s


playoff_name_col = find_name_column(playoff_stats, 'matched_playoff_stats_2025.csv')
salary_name_col = find_name_column(regular_salary, 'regular_salary.csv')

if 'Salary' not in regular_salary.columns:
    raise ValueError(f"'Salary' column not found in regular_salary.csv. Columns: {list(regular_salary.columns)}")

# Build normalized name keys for matching
playoff_stats = playoff_stats.copy()
regular_salary = regular_salary.copy()
playoff_stats['name_key'] = playoff_stats[playoff_name_col].apply(normalize_player_name)
regular_salary['name_key'] = regular_salary[salary_name_col].apply(normalize_player_name)

salary_lookup = (
    regular_salary[[salary_name_col, 'Salary', 'name_key']]
    .dropna(subset=['name_key'])
    .drop_duplicates(subset=['name_key'])
    .rename(columns={salary_name_col: 'Regular_Player'})
)

# Match salary onto playoff stats by normalized player name
playoff_with_salary = playoff_stats.merge(
    salary_lookup[['name_key', 'Regular_Player', 'Salary']],
    on='name_key',
    how='left'
)

playoff_with_salary = playoff_with_salary.drop(columns=['name_key'])

# Save final dataset
playoff_with_salary.to_csv('data/matched_playoff_stats_with_salary_2025.csv', index=False)

matched_salary_rows = playoff_with_salary['Salary'].notna().sum()
total_rows = len(playoff_with_salary)

print(f'Playoff name column: {playoff_name_col}')
print(f'Regular salary name column: {salary_name_col}')
print(f'Salary matched rows: {matched_salary_rows}/{total_rows}')
print('Saved file: data/matched_playoff_stats_with_salary_2025.csv')
print('\nPreview:')
print(playoff_with_salary.head(10).to_string(index=False))

Playoff name column: Player
Regular salary name column: Player
Salary matched rows: 121/121
Saved file: data/matched_playoff_stats_with_salary_2025.csv

Preview:
                  Player Pos  Age  Tm  G  GS   MP   FG  FGA   FG%  3P  3PA   3P%   2P  2PA   2P%  eFG%  FT  FTA   FT%  ORB  DRB  TRB  AST  STL  BLK  TOV  PF  PTS           Regular_Player   Salary
            Steven Adams   C   31 HOU  7   0 22.1  1.7  2.9 0.600 0.0  0.0   NaN  1.7  2.9 0.600 0.600 2.3  4.3 0.533  3.3  3.3  6.6  0.6  0.4  1.1  0.7 1.0  5.7             Steven Adams 15000000
             Bam Adebayo   C   27 MIA  4   4 38.3  7.0 16.0 0.438 1.8  5.3 0.333  5.3 10.8 0.488 0.492 1.8  2.8 0.636  4.0  7.0 11.0  4.3  1.0  0.3  3.0 2.3 17.5              Bam Adebayo 49800000
            Santi Aldama  PF   24 MEM  4   1 30.5  5.3 11.0 0.477 2.5  6.0 0.417  2.8  5.0 0.550 0.591 0.0  0.0   NaN  0.5  5.5  6.0  1.8  0.0  0.3  1.3 1.8 13.0             Santi Aldama 19375000
Nickeil Alexander-Walker  SG   26 MIN 15   0 20.7  2.9

In [10]:
import re
import pandas as pd

# Load playoff salary data and the enriched regular-season salary data
playoff_df = pd.read_csv('data/matched_playoff_stats_with_salary_2025.csv', encoding='latin1')
regular_df = pd.read_csv('data/new_regular_salary.csv', encoding='latin1')


def find_name_column(df, label):
    candidates = [
        'Player', 'PLAYER', 'player',
        'Regular_Player', 'regular_player',
        'Name', 'NAME', 'name',
        'Player Name', 'player_name', 'PlayerName'
    ]
    for c in candidates:
        if c in df.columns:
            return c
    for c in df.columns:
        low = str(c).strip().lower()
        if 'player' in low or low == 'name':
            return c
    raise ValueError(f'Could not find player-name column in {label}. Columns: {list(df.columns)}')


def normalize_player_name(name):
    if pd.isna(name):
        return ''
    s = str(name).lower().strip()
    s = re.sub(r"\b(jr|sr|ii|iii|iv|v)\b\.?", '', s)
    s = re.sub(r"[^a-z\s]", '', s)
    s = re.sub(r"\s+", ' ', s).strip()
    return s


playoff_name_col = find_name_column(playoff_df, 'matched_playoff_stats_with_salary_2025.csv')
regular_name_col = find_name_column(regular_df, 'new_regular_salary.csv')

# Columns to bring over from the regular salary data
feature_cols = ['Awards', 'Pos_C', 'Pos_PF', 'Pos_PG', 'Pos_SF', 'Pos_SG']
missing_features = [c for c in feature_cols if c not in regular_df.columns]
if missing_features:
    raise ValueError(f'Missing expected feature columns in new_regular_salary.csv: {missing_features}')

# Normalize names for matching
playoff_df = playoff_df.copy()
regular_df = regular_df.copy()
playoff_df['name_key'] = playoff_df[playoff_name_col].apply(normalize_player_name)
regular_df['name_key'] = regular_df[regular_name_col].apply(normalize_player_name)

# Build lookup table with one row per player
lookup_cols = ['name_key', regular_name_col] + feature_cols
feature_lookup = (
    regular_df[lookup_cols]
    .dropna(subset=['name_key'])
    .drop_duplicates(subset=['name_key'])
    .rename(columns={regular_name_col: 'Regular_Player_Source'})
)

# Merge the award/position features onto the playoff salary file
playoff_enriched = playoff_df.merge(
    feature_lookup,
    on='name_key',
    how='left',
    suffixes=('', '_regular')
)

# Clean up helper columns
playoff_enriched = playoff_enriched.drop(columns=['name_key'])

# Save the updated file
output_path = 'data/matched_playoff_stats_with_salary_and_features_2025.csv'
playoff_enriched.to_csv(output_path, index=False)

matched_feature_rows = playoff_enriched[feature_cols].notna().all(axis=1).sum()
print(f'Playoff name column: {playoff_name_col}')
print(f'Regular salary name column: {regular_name_col}')
print(f'Rows with matched feature columns: {matched_feature_rows}/{len(playoff_enriched)}')
print(f'Saved file: {output_path}')
print('\nPreview:')
preview_cols = [playoff_name_col, 'Regular_Player', 'Salary'] + feature_cols
preview_cols = [c for c in preview_cols if c in playoff_enriched.columns]
print(playoff_enriched[preview_cols].head(10).to_string(index=False))

Playoff name column: Player
Regular salary name column: Player
Rows with matched feature columns: 121/122
Saved file: data/matched_playoff_stats_with_salary_and_features_2025.csv

Preview:
                  Player           Regular_Player   Salary  Awards  Pos_C  Pos_PF  Pos_PG  Pos_SF  Pos_SG
            Steven Adams             Steven Adams 15000000     0.0    1.0     0.0     0.0     0.0     0.0
             Bam Adebayo              Bam Adebayo 49800000     1.0    1.0     0.0     0.0     0.0     0.0
            Santi Aldama             Santi Aldama 19375000     0.0    0.0     1.0     0.0     0.0     0.0
Nickeil Alexander-Walker Nickeil Alexander-Walker 16666667     0.0    0.0     0.0     0.0     0.0     1.0
           Jarrett Allen            Jarrett Allen 28000000     0.0    1.0     0.0     0.0     0.0     0.0
           Kyle Anderson            Kyle Anderson 10000000     0.0    0.0     1.0     0.0     0.0     0.0
   Giannis Antetokounmpo    Giannis Antetokounmpo 54126450     1.0   

In [12]:
import pandas as pd
import statsmodels.api as sm

# Load data (file contains non-UTF8 player name characters)
df = pd.read_csv('data/matched_playoff_stats_with_salary_2025.csv', encoding='latin1')

# Independent variables requested
x_cols = [
    'GS', 'MP', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PTS'
]

y_col = 'Salary'

# Keep only needed columns and coerce to numeric
model_df = df[x_cols + [y_col]].copy()
model_df = model_df.apply(pd.to_numeric, errors='coerce').dropna()

# Convert salary to millions
model_df['Salary_millions'] = model_df[y_col] / 1_000_000

X = sm.add_constant(model_df[x_cols])
y = model_df['Salary_millions']

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:        Salary_millions   R-squared:                       0.740
Model:                            OLS   Adj. R-squared:                  0.721
Method:                 Least Squares   F-statistic:                     40.13
Date:                Sun, 12 Apr 2026   Prob (F-statistic):           1.31e-29
Time:                        20:48:38   Log-Likelihood:                -424.15
No. Observations:                 122   AIC:                             866.3
Df Residuals:                     113   BIC:                             891.5
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          4.7323      2.282      2.074      0.0

In [13]:
import pandas as pd
import statsmodels.api as sm

# Load data (file contains non-UTF8 player name characters)
df = pd.read_csv('data/playoff_salary.csv', encoding='latin1')

# Independent variables requested
x_cols = [
    'GS', 'MP', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PTS',
    'Awards', 'Pos_C', 'Pos_PF', 'Pos_PG', 'Pos_SF', 'Pos_SG'
]

y_col = 'Salary'

# Keep only needed columns and coerce to numeric
model_df = df[x_cols + [y_col]].copy()
model_df = model_df.apply(pd.to_numeric, errors='coerce').dropna()

# Convert salary to millions
model_df['Salary_millions'] = model_df[y_col] / 1_000_000

X = sm.add_constant(model_df[x_cols])
y = model_df['Salary_millions']

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:        Salary_millions   R-squared:                       0.776
Model:                            OLS   Adj. R-squared:                  0.749
Method:                 Least Squares   F-statistic:                     28.75
Date:                Sun, 12 Apr 2026   Prob (F-statistic):           3.52e-29
Time:                        20:49:08   Log-Likelihood:                -415.03
No. Observations:                 122   AIC:                             858.1
Df Residuals:                     108   BIC:                             897.3
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          4.4467      1.862      2.388      0.0

In [14]:
import pandas as pd
import statsmodels.api as sm

# Load data (file contains non-UTF8 player name characters)
df = pd.read_csv('data/playoff_salary.csv', encoding='latin1')

# Independent variables requested
x_cols = [
    'GS', 'MP', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PTS',
    'Awards', 'Pos_C', 'Pos_PF', 'Pos_SF', 'Pos_SG'
]

y_col = 'Salary'

# Keep only needed columns and coerce to numeric
model_df = df[x_cols + [y_col]].copy()
model_df = model_df.apply(pd.to_numeric, errors='coerce').dropna()

# Convert salary to millions
model_df['Salary_millions'] = model_df[y_col] / 1_000_000

X = sm.add_constant(model_df[x_cols])
y = model_df['Salary_millions']

model = sm.OLS(y, X).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:        Salary_millions   R-squared:                       0.776
Model:                            OLS   Adj. R-squared:                  0.749
Method:                 Least Squares   F-statistic:                     28.75
Date:                Sun, 12 Apr 2026   Prob (F-statistic):           3.52e-29
Time:                        21:08:59   Log-Likelihood:                -415.03
No. Observations:                 122   AIC:                             858.1
Df Residuals:                     108   BIC:                             897.3
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          5.5991      3.411      1.642      0.1